In [ ]:
#test_landsat_geemap.ipynb tests Landsat with geemap
#NOTE: this is a clean copy of part of what I'm trying to do in GD_Landsat_02_Download.ipynb
#TODO: fix display issues/understand scaling. See also: test_landsat_histogram
#Grok prompts: 
#geemap code to view landsat imagery
#this is good so far but the time slider gives ValueError: The length of labels must be equal to the number of images in the ImageCollection.
#add earlier landsat satellites

# ------------------------------------------------------------
# 1. Install geemap (uncomment if running in a fresh environment)
# ------------------------------------------------------------
# !pip install -q geemap

# ------------------------------------------------------------
# 2. Import libraries
# ------------------------------------------------------------
import ee
import geemap

# ------------------------------------------------------------
# 3. Authenticate & initialize Earth Engine
# ------------------------------------------------------------
# The first time you run this you will be prompted to authenticate.
ee.Authenticate()
ee.Initialize()

# ------------------------------------------------------------
# 4. Define an Area of Interest (AOI)
# ------------------------------------------------------------
# Example: a rectangle around San Francisco, CA
# Change the coordinates to any geometry you like.
aoi = ee.Geometry.Rectangle([-122.55, 37.68, -122.35, 37.82])

In [ ]:
# ------------------------------------------------------------
# 5. Load Landsat Collections (L4, L5, L7, L8, L9 - Collection 2, Tier 1 SR)
# ------------------------------------------------------------
#landsat = (
#    ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')   # Landsat 8
#    .merge(ee.ImageCollection('LANDSAT/LC09/C02/T1_L2'))  # Landsat 9
#    .filterBounds(aoi)
#    .filterDate('2023-01-01', '2024-12-31')      # adjust date range
#)
landsat4 = ee.ImageCollection('LANDSAT/LT04/C02/T1_L2')   # Landsat 4 TM (1982-1993)
landsat5 = ee.ImageCollection('LANDSAT/LT05/C02/T1_L2')   # Landsat 5 TM (1984-2013)
landsat7 = ee.ImageCollection('LANDSAT/LE07/C02/T1_L2')   # Landsat 7 ETM+ (1999-present)
landsat8 = ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')      # Landsat 8 (2013-present)
landsat9 = ee.ImageCollection('LANDSAT/LC09/C02/T1_L2')  # Landsat 9 (2021-present)


landsat = landsat4.merge(landsat5).merge(landsat7).merge(landsat8).merge(landsat9) \
    .filterBounds(aoi) \
    .filterDate('1982-01-01', '2024-12-31')
landsat #2299 elements

In [ ]:
landsat4TOA = ee.ImageCollection('LANDSAT/LT04/C02/T1_TOA')
landsat5TOA = ee.ImageCollection('LANDSAT/LT05/C02/T1_TOA')
landsat7TOA = ee.ImageCollection('LANDSAT/LE07/C02/T1_TOA')
landsat8TOA = ee.ImageCollection('LANDSAT/LC08/C02/T1_TOA')
landsat9TOA = ee.ImageCollection('LANDSAT/LC09/C02/T1_TOA')

landsatTOA = (
    landsat4TOA.merge(landsat5TOA).merge(landsat7TOA).merge(landsat8TOA).merge(landsat9TOA) \
    .filterBounds(aoi) \
    .filterDate('1982-01-01', '2024-12-31')
)

In [ ]:
landsatTOA #2363 elements 

In [ ]:
# ------------------------------------------------------------
# 6. Cloud masking function (CFMask band)
# ------------------------------------------------------------
def mask_clouds(image):
    qa = image.select('QA_PIXEL')
    # Bits 3 & 5: cloud (bit 3) and cloud shadow (bit 5)
    cloud_bit = 1 << 3
    shadow_bit = 1 << 5
    mask = qa.bitwiseAnd(cloud_bit).eq(0) \
              .And(qa.bitwiseAnd(shadow_bit).eq(0))
    return image.updateMask(mask)

# ------------------------------------------------------------
# 7. Apply scaling factors (Landsat C2 SR is stored as DN*0.000027 + -0.2)
# ------------------------------------------------------------
def apply_scale_factors(image):
    optical = image.select('SR_B.').multiply(0.000027).add(-0.2)
    #thermal = image.select('ST_B10').multiply(0.00341802).add(149.0) #this was giving problems, also could be 'ST_B.*'
    return image.addBands(optical, None, True) #\
                #.addBands(thermal, None, True)

# ------------------------------------------------------------
# 7a. Apply TOA scaling: DN * 0.000027 + -0.2
# ------------------------------------------------------------
def apply_toa_scale(image):
    # Select optical bands (B1 to B7 for TM/ETM, B2 to B7 for OLI)
    optical_bands = image.select(['B[1-7]'])  # Regex: B1 to B7
    scaled = optical_bands.multiply(0.000027).add(-0.2)
    # Thermal (optional): B6 (TM/ETM), B10 (OLI)
    #thermal = image.select(['B6', 'B10']).multiply(0.00341802).add(149.0) #this was giving problems
    return image.addBands(scaled, None, True)#.addBands(thermal, None, True)

In [ ]:
# ------------------------------------------------------------
# 8. Build median composite (multi-decadal, cloud-free)
# ------------------------------------------------------------
compositeTOA = (
    landsatTOA
    .map(mask_clouds)
    .map(apply_toa_scale)
    .median()
    .clip(aoi)
)
compositeTOA

In [ ]:
# ------------------------------------------------------------
# 8. Build a median composite (cloud-free)
# ------------------------------------------------------------
composite = (
    landsat
    .map(mask_clouds)
    .map(apply_scale_factors)
    .median()
    .clip(aoi)
)
composite

In [ ]:
# ------------------------------------------------------------
# 9. Visualization parameters
# ------------------------------------------------------------
vis_true = {
    'bands': ['SR_B4', 'SR_B3', 'SR_B2'],   # Red, Green, Blue
    'min': 0.0,
    'max': 0.3,
    'gamma': 1.4
}

vis_false = {
    'bands': ['SR_B5', 'SR_B4', 'SR_B3'],   # NIR, Red, Green
    'min': 0.0,
    'max': 0.3,
    'gamma': 1.4
}

vis_trueTOA = {
    'bands': ['B4', 'B3', 'B2'],  # Red, Green, Blue
    'min': 0.0,
    'max': 0.4,  # Slightly higher than SR for bright scenes
    'gamma': 1.3
}

vis_falseTOA = {
    'bands': ['B5', 'B4', 'B3'],  # NIR, Red, Green
    'min': 0.0,
    'max': 0.5,
    'gamma': 1.3
}
# ------------------------------------------------------------
# 10. Create the interactive map
# ------------------------------------------------------------
Map = geemap.Map(center=[37.75, -122.45], zoom=11)
Map.addLayer(composite, vis_true, 'Landsat True Color')
Map.addLayer(composite, vis_false, 'Landsat False Color (NIR)')
Map.addLayer(compositeTOA, vis_trueTOA, 'Landsat TOA True')
Map.addLayer(compositeTOA, vis_falseTOA, 'Landsat TOA False')

In [ ]:
Map.addLayer(aoi,{},'aoi')
Map

In [ ]:
# Optional: add a layer control & legend
#geemap already has one, don't need an extra one: Map.addLayerControl()
#Map.add_legend(title='Landsat Composite', builtin_legend='Landsat') #gives error

In [ ]:
# ------------------------------------------------------------
# 11. Prepare collection for time slider
# ------------------------------------------------------------
collection = landsat.map(mask_clouds).map(apply_scale_factors) #.filter(ee.Filter.lt('CLOUD_COVER', 20))  # <20% clouds to reduce image count
dates=collection.aggregate_array('DATE_ACQUIRED').getInfo()

# Print number of images (uncomment to check)
print('Number of images:', collection.size().getInfo())

# ------------------------------------------------------------
# 12. Add time slider (FIXED: Omit labels for auto-generation)
# ------------------------------------------------------------
#Map.add_time_slider(collection,vis_true,labels=dates,time_interval=1,layer_name='Landsat Time Series')
Map.add_time_slider(
    collection,
    vis_true,
    # labels=None,  # Auto-generates from image properties (e.g., dates)
    time_interval=2,
    layer_name='Landsat Time Series'
)

# ALTERNATIVE: Custom labels (uncomment if needed)
# indices = collection.aggregate_array('system:index').getInfo()
# labels = [f'Image {i+1}: {idx[:8]}' for i, idx in enumerate(indices)]  # YYYYMMDD
# Map.add_time_slider(
#     collection,
#     vis_true,
#     labels=labels,
#     time_interval=3,
#     layer_name='Landsat Time Series (1982–Present)'
# )

In [ ]:
composite

In [ ]:
# ------------------------------------------------------------
# 13. Display the map
# ------------------------------------------------------------
Map